In [ ]:
!pip install groq -q

In [ ]:
import sqlite3
import pandas as pd
import os

from groq import Groq
import re

print("All Libraries imported successfully")

All Libraries imported successfully


In [ ]:
import os
os.environ["GROQ_API_KEY"]=""
client=Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"
print("Groq client initialized successfully")
print(f"Using model:{MODEL}")

Groq client initialized successfully
Using model:llama-3.1-8b-instant


In [ ]:
import io

csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df=pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print("\nTable")
df.head(30)

Dataset loaded: 30 rows, 8 columns

Table


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+
5,6,Divya Krishnan,20,Female,Science,83,88,A
6,7,Karan Singh,22,Male,Mathematics,74,81,B
7,8,Ananya Gupta,21,Female,Programming,89,96,A
8,9,Vikram Reddy,20,Male,Science,70,79,B
9,10,Pooja Sharma,22,Female,Mathematics,55,72,D


In [ ]:
conn=sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)
print("Database created: College.db")
print("Table 'students' created with 30 student records")
test_df=pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)
print(f"\nVerification: {test_df['total_rows'][0]}rows in database")

Database created: College.db
Table 'students' created with 30 student records

Verification: 30rows in database


In [ ]:
def get_schema(conn, table_name="students"):

  cursor=conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns=cursor.fetchall()

  schema_lines=[f"Table: {table_name}"]
  schema_lines.append("Columns:")

  for col in columns:
    schema_lines.append(f"  - {col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows=cursor.fetchall()
  schema_lines.append("\nSample rows (first 3):")

  for row in sample_rows:
    schema_lines.append(f"   {row}")

  return "\n".join(schema_lines)

schema=get_schema(conn)
print(schema)

Table: students
Columns:
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - subject (TEXT)
  - marks (INTEGER)
  - attendance (INTEGER)
  - grade (TEXT)

Sample rows (first 3):
   (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
   (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
   (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question,schema_text,client,model):
  system_prompt = f"""You are an expert SQL assistant.
You are connected to a SQLite database with the following structure:


{schema_text}


Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or text — only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
4. The table name is: students
5. Only use column names that exist in the schema above.
6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
"""

  response=client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_question}
      ],
      temperature=0.0
  )
  return response.choices[0].message.content.strip()
  return sql_query
quetion="show me all male student with grage b and from programming"
print(f"Question: {quetion}")
sql_query=generate_sql(quetion,schema,client,Model)
print(f"SQL Query: \n{sql_query}")

Question: show me all male student with grage b and from programming
SQL Query: 
SELECT * FROM students WHERE gender = 'Male' AND grade = 'B' AND subject = 'Programming'


In [ ]:
def execute_sql(sql_query, conn):
  clean_sql = sql_query.strip()
  clean_sql = re.sub(r'```sql\s*', '', clean_sql)
  clean_sql = re.sub(r'```s*', '', clean_sql)
  clean_sql = clean_sql.strip()
  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None
  except Exception as e:
    return None, str(e)
print(f"Executing SQL: {sql_query}")
result, error = execute_sql(sql_query, conn)
if error:
  print(f"Error executing SQL: {error}")
else:
  print(f"\nQuery Returned {len(result)} rows:")
  print(result)

Executing SQL: SELECT * FROM students WHERE gender = 'Male' AND grade = 'B' AND subject = 'Programming'

Query Returned 2 rows:
   student_id           name  age gender      subject  marks  attendance grade
0          17   Manish Joshi   21   Male  Programming     73          82     B
1          29  Ravi Krishnan   21   Male  Programming     78          88     B


In [ ]:
def text_to_sql_agent(user_question, conn, client, model,verbose):

  print("="*60)

  print(f"USER QUESTION: {user_question}")
  print("="*60)

  if verbose:
    print("\n[STEP 1] Reading database schema...")
    schema_text=get_schema(conn)

  if verbose:
    print("Schema loaded successfully")

  if verbose:
    print("\n[STEP 2] Generating SQL query with Groq LLM...")

  generated_sql=generate_sql(user_question,schema_text,client,model)

  if verbose:
    print(f"Generated SQL:\n {generated_sql}")

  if verbose:
    print("\n[STEP 3] Executing SQL on the database...")

  result_df,error=execute_sql(generated_sql,conn)

  if error:
    print(f"SQL Execution Error: {error}")
    return None,generated_sql

  if verbose:
    print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
    print("\nRESULTS:")
    print("-"*40)
    print(result_df.to_string(index=False))

  print("="*60)

  return result_df,generated_sql

result,sql_used=text_to_sql_agent(
    "Show top 5 students in Programming",
    conn,client,MODEL,True
)

USER QUESTION: Show top 5 students in Programming

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
         11 Aditya Kumar   21   Male Programming     97          99    A+
          3  Rohan Mehta   20   Male Programming     95          98    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
          5   Arjun Nair   21   Male Programming     91          94    A+


In [ ]:
result1,_= text_to_sql_agent(
    "show me all student who study Mathematics",
    conn,client,MODEL,True
)

USER QUESTION: show me all student who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE subject = 'Mathematics'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Fe

In [ ]:
result1,_= text_to_sql_agent(
    "show me all student filtering by age 20 & 21",
    conn,client,MODEL,True
)

USER QUESTION: show me all student filtering by age 20 & 21

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE age = 20 OR age = 21

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 21 row(s)

RESULTS:
----------------------------------------
 student_id           name  age gender     subject  marks  attendance grade
          1   Aarav Sharma   20   Male Mathematics     88          92     A
          2    Priya Patel   21 Female     Science     76          85     B
          3    Rohan Mehta   20   Male Programming     95          98    A+
          5     Arjun Nair   21   Male Programming     91          94    A+
          6 Divya Krishnan   20 Female     Science     83          88     A
          8   Ananya Gupta   21 Female Programming     89          96     A
          9   Vikram Reddy   20   Male     Science     70          79     B
         11   Aditya Kuma

In [ ]:
result1,_= text_to_sql_agent(
    "count all stdents in  Mathematics annd their marks who all got the A+ grade and show the students name",
    conn,client,MODEL,True
)

USER QUESTION: count all stdents in  Mathematics annd their marks who all got the A+ grade and show the students name

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT COUNT(student_id), name FROM students WHERE subject = 'Mathematics' AND grade = 'A+'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 COUNT(student_id)          name
                 1 Swathi Pillai


In [ ]:
!pip install crewai --force-reinstall litellm==1.30.0 --force-reinstall requests python-dotenv apscheduler --q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
wandb 0.27.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-api>=1.35.0, but you have opentelemetry-api 1.34.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
bigframes 2.40.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
transformers 5.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.
gr

In [ ]:
import os
os.environ["GROQ_API_KEY"]=''
print("Groq api initialized succesfully")

Groq api initialized succesfully


In [ ]:
!pip install 'crewai[litellm]' --upgrade --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.2.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have 

In [ ]:
from crewai import Agent,Task, Crew

def research_agent_example():
  researcher= Agent(
    role='Research Analyst',
    goal='Identify top AI trends for 2025',
    backstory='Experienced technology researcher with 10+ years of industry experience',
    verbose=True,
    llm='gqoq/llama-3.3-70b-versatile'
)

  research_task=Task(
      desciption='List and analyze the top 3 AI trends for 2025 with examples',
      expected_output='Comprehensive report with detailed explanation for each trend',
      agent=researcher
  )
  crew=crew(
      agents=[researcher],
      tasks=[research_task],
      verbose=True
  )
  return crew.kickoff()
if __name__=="__main__":
  result=research_agent_example()
  print(result)

ERROR:crewai.llm:Unable to initialize LLM with model 'gqoq/llama-3.3-70b-versatile'. The model did not match any supported native provider (openai, anthropic, claude, azure, azure_openai, google, gemini, bedrock, aws, openrouter, deepseek, ollama, ollama_chat, hosted_vllm, cerebras, dashscope), and the LiteLLM fallback package is not installed.

To fix this, either:
  1. Install LiteLLM for broad model support: uv add 'crewai[litellm]'
or
pip install litellm

For more details, see: https://docs.crewai.com/en/learn/llm-connections
ERROR:crewai.utilities.llm_utils:Error instantiating LLM from string: Unable to initialize LLM with model 'gqoq/llama-3.3-70b-versatile'. The model did not match any supported native provider (openai, anthropic, claude, azure, azure_openai, google, gemini, bedrock, aws, openrouter, deepseek, ollama, ollama_chat, hosted_vllm, cerebras, dashscope), and the LiteLLM fallback package is not installed.

To fix this, either:
  1. Install LiteLLM for broad model suppo

ImportError: Unable to initialize LLM with model 'gqoq/llama-3.3-70b-versatile'. The model did not match any supported native provider (openai, anthropic, claude, azure, azure_openai, google, gemini, bedrock, aws, openrouter, deepseek, ollama, ollama_chat, hosted_vllm, cerebras, dashscope), and the LiteLLM fallback package is not installed.

To fix this, either:
  1. Install LiteLLM for broad model support: uv add 'crewai[litellm]'
or
pip install litellm

For more details, see: https://docs.crewai.com/en/learn/llm-connections